# Audio Scene Narrator

**Sound event detection + foreground/background analysis + LLM reasoning, with a small benchmark against an end-to-end audio-LLM.**

1. A pretrained detector (PANNs CNN14, AudioSet) finds sound events with time stamps.
2. Each event is labelled **foreground** or **background** from its level contrast and how much of the clip it covers.
3. An instruction-tuned LLM (Qwen2.5-3B) reads that timeline and describes the scene or answers questions.
4. On synthetic scenes with known ground truth (built from ESC-50), the cascade is compared with **Qwen2-Audio-7B**, which listens to the audio directly.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. A full run takes about 30-40 minutes, most of it model downloads.

## 1. Get the code and install dependencies

In [ ]:
import os
# Option A: upload audio-scene-narrator.zip when the file picker appears.
if not os.path.exists("/content/audio-scene-narrator"):
    from google.colab import files
    files.upload()
    !unzip -q -o /content/audio-scene-narrator.zip -d /content
# Option B: once the repo is on your GitHub, replace the lines above with:
# !git clone https://github.com/<your-username>/audio-scene-narrator.git /content/audio-scene-narrator
%cd /content/audio-scene-narrator
!pip install -q -r requirements.txt
!nvidia-smi -L

## 2. Build benchmark scenes from ESC-50
Each 10 s scene has one steady background sound and 1-3 foreground events. Because we mix them ourselves, we know the exact ground truth. ESC-50 is about 600 MB.

In [ ]:
import json
from IPython.display import Audio, display
from scene_narrator.synth import build_scenes

if not os.path.exists("/content/ESC-50-master"):
    !wget -q https://github.com/karolpiczak/ESC-50/archive/master.zip -O /content/esc50.zip
    !unzip -q /content/esc50.zip -d /content

scenes = build_scenes("/content/ESC-50-master", "/content/scenes", n_scenes=30, seed=0)
print(json.dumps(scenes[0]["events"], indent=1))
display(Audio(scenes[0]["path"]))

## 3. Load the detector and the LLM

In [ ]:
from scene_narrator.tagger import PannsTagger
from scene_narrator.narrator import HFChatLLM, Narrator
from scene_narrator.pipeline import SceneNarrator

tagger = PannsTagger()
llm = HFChatLLM("Qwen/Qwen2.5-3B-Instruct")
pipe = SceneNarrator(tagger, Narrator(llm))

## 4. Try it on one scene

In [ ]:
from scene_narrator.events import timeline_to_text
import matplotlib.pyplot as plt
from scene_narrator.plotting import plot_timeline

s = scenes[0]
scene = pipe.analyze(s["path"])
print(timeline_to_text(scene))
fig = plot_timeline(scene, title=s["id"]); display(fig); plt.close(fig)
print("\nGround truth:", [(e["class"], e["role"], e["onset"], e["offset"]) for e in s["events"]])
print("\nDescription:\n", pipe.describe(scene))
print("\nQ: What stands out from the background, and when?\nA:", pipe.ask(scene, "What stands out from the background, and when?"))

## 5. Interactive demo
Opens a public Gradio link (valid for 72 h) that you can show on a laptop or phone. Upload your own recordings or record with the microphone. Run this before step 7, which unloads the LLM.

In [ ]:
from app import build_demo
demo = build_demo(pipe, examples=[s["path"] for s in scenes[:4]])
demo.launch(share=True, debug=False)

## 6. Evaluate the cascade
* **Detection:** for every ground-truth sound, was it detected (recall) and was its role right?
* **Question answering:** five templated question types with known answers:
  `presence_yes`, `presence_no` (a hallucination check), `background`, `role` (foreground vs. background) and `order` (before/after).
* **Control:** the same LLM without any detector output, to show how much the timeline actually contributes.

In [ ]:
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from scene_narrator.synth import make_questions
from scene_narrator.evaluate import match_events, detection_summary, score_answer, qa_summary

questions = [q for s in scenes for q in make_questions(s, np.random.default_rng(int(s["id"][-3:])))]
paths = {s["id"]: s["path"] for s in scenes}
print(len(questions), "questions over", len(scenes), "scenes")

timelines, det_rows = {}, []
for s in tqdm(scenes, desc="detection"):
    a = pipe.analyze(s["path"])
    timelines[s["id"]] = timeline_to_text(a)
    det_rows += match_events(s, a)
det_summary = detection_summary(det_rows)
display(det_summary)

rows = []
for q in tqdm(questions, desc="cascade QA"):
    ans = pipe.narrator.answer(timelines[q["scene"]], q["question"])
    rows.append({**q, "system": "cascade: PANNs + Qwen2.5-3B", "response": ans, "correct": score_answer(q, ans)})
    blind = pipe.narrator.answer("No detector output available.", q["question"])
    rows.append({**q, "system": "control: LLM without audio", "response": blind, "correct": score_answer(q, blind)})
display(qa_summary(pd.DataFrame(rows)))

## 7. End-to-end baseline: Qwen2-Audio-7B (4-bit)
Frees the cascade's LLM first so both models fit on a T4. To use the demo again afterwards, re-run step 3.

In [ ]:
import gc, torch
pipe.narrator = None
del llm
gc.collect(); torch.cuda.empty_cache()

from scene_narrator.audio_llm import Qwen2AudioQA
alm = Qwen2AudioQA("Qwen/Qwen2-Audio-7B-Instruct", load_in_4bit=True)
for q in tqdm(questions, desc="audio-LLM QA"):
    ans = alm.ask(paths[q["scene"]], q["question"])
    rows.append({**q, "system": "end-to-end: Qwen2-Audio-7B", "response": ans, "correct": score_answer(q, ans)})

## 8. Results

In [ ]:
import matplotlib.pyplot as plt
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(rows)
summary = qa_summary(df)
display(summary)

order = ["presence_yes", "presence_no", "background", "role", "order", "overall"]
colors = {"cascade: PANNs + Qwen2.5-3B": "#d9622b", "end-to-end: Qwen2-Audio-7B": "#3b7dbf",
          "control: LLM without audio": "#b0b0b0"}
plot_df = summary[[c for c in order if c in summary.columns]].T
ax = plot_df.plot.bar(figsize=(10, 4), rot=0, ylim=(0, 1.05),
                      color=[colors.get(c, "#666") for c in plot_df.columns], width=0.8)
ax.set_ylabel("accuracy"); ax.set_xlabel(""); ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"Scene QA accuracy on {len(scenes)} synthetic scenes ({len(questions)} questions)", loc="left")
plt.tight_layout(); plt.savefig("results/qa_accuracy.png", dpi=150); plt.show()

df.to_csv("results/qa_answers.csv", index=False)
summary.to_csv("results/qa_summary.csv")
det_summary.to_csv("results/detection_summary.csv")
pd.DataFrame(det_rows).to_csv("results/detection_events.csv", index=False)
print("Saved to results/")

## 9. Error analysis: where do the two systems disagree?

In [ ]:
pd.set_option("display.max_colwidth", 120)
wide = df.pivot_table(index=["scene", "type", "question", "answer"], columns="system",
                      values="correct", aggfunc="first").reset_index()
casc, e2e = "cascade: PANNs + Qwen2.5-3B", "end-to-end: Qwen2-Audio-7B"
diff = wide[wide[casc] != wide[e2e]]
print(len(diff), "questions where only one system is right")
display(diff.head(15))

# Look closely at one case: what the detector saw vs. what each system answered
if len(diff):
    r = diff.iloc[0]
    print(timelines[r["scene"]])
    display(df[(df.scene == r["scene"]) & (df.question == r["question"])][["system", "response", "correct"]])
    display(Audio(paths[r["scene"]]))

## Next steps
* **Plug in your own foreground/background model:** replace `foreground_score` in `scene_narrator/events.py`.
* **Harder scenes:** overlapping events, lower SNR, real recordings (e.g. DCASE or TAU Urban Acoustic Scenes).
* **Richer questions:** counting ("how many times does the dog bark?"), durations and relations between sounds.
* **Fix the weak spots:** better label grounding, uncertainty-aware prompting, or fine-tuning an audio-LLM on foreground/background supervision.